# 强化学习算法对比实验

## Standard DQN vs Double DQN

本实验对比标准 DQN 与 Double DQN 在多智能体配送调度任务中的表现。

### 实验目标
- 比较两种算法的收敛速度
- 评估订单完成率和配送效率
- 分析 Double DQN 对过估计问题的改善效果

---

## 1. 环境设置与导入

In [5]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# 添加项目根目录到路径（Notebook 兼容）
notebook_dir = Path(os.getcwd())
if notebook_dir.name == "experiments":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rl_agents.training_manager import TrainingManager

print("✓ 导入完成")
print(f"当前目录: {os.getcwd()}")
print(f"项目根目录: {project_root}")

ModuleNotFoundError: No module named 'rl_agents'

## 2. 实验配置

### 环境参数
- **网格大小**: 12×12
- **车辆数量**: 3 辆
- **最大步数**: 200 步/episode
- **订单数量**: 15 个/episode

### 训练参数
- **训练轮数**: 300 episodes
- **评估间隔**: 每 20 episodes
- **保存间隔**: 每 100 episodes
- **早停阈值**: 成功率 85%

In [6]:
# 环境配置
env_config = {
    "grid_size": 12,
    "num_cars": 3,
    "max_steps": 200,
    "max_orders_per_episode": 15
}

# 训练配置
training_config = {
    "max_episodes": 300,
    "eval_interval": 20,
    "save_interval": 100,
    "early_stop_threshold": 0.85,
    "patience": 150,
}

print("✓ 配置参数设置完成")
print(f"环境: {env_config}")
print(f"训练: {training_config}")

✓ 配置参数设置完成
环境: {'grid_size': 12, 'num_cars': 3, 'max_steps': 200, 'max_orders_per_episode': 15}
训练: {'max_episodes': 300, 'eval_interval': 20, 'save_interval': 100, 'early_stop_threshold': 0.85, 'patience': 150}


## 3. 训练标准 DQN

标准 DQN 使用同一个网络进行动作选择和Q值估计，可能存在过估计问题。

In [7]:
print("=" * 80)
print("[1/2] 训练标准 DQN...")
print("=" * 80)

# 标准 DQN 配置
dqn_agent_config = {
    "use_double_dqn": False,  # 标准 DQN
}

# 创建训练器
trainer_dqn = TrainingManager(
    agent_type="DQN",
    environment_config=env_config,
    training_config=training_config,
    agent_config=dqn_agent_config,
    save_dir="experiments/results/standard_dqn",
)

# 开始训练
results_dqn = trainer_dqn.train_agent()

print("\n✓ 标准 DQN 训练完成")
print(f"训练轮数: {len(results_dqn['episode_rewards'])}")
print(f"最终奖励: {results_dqn['episode_rewards'][-1]:.2f}")

[1/2] 训练标准 DQN...


NameError: name 'TrainingManager' is not defined

## 4. 训练 Double DQN

Double DQN 使用两个网络解耦动作选择和Q值估计，减少过估计偏差。

In [ ]:
print("=" * 80)
print("[2/2] 训练 Double DQN...")
print("=" * 80)

# Double DQN 配置
double_dqn_agent_config = {
    "use_double_dqn": True,  # Double DQN
}

# 创建训练器
trainer_double_dqn = TrainingManager(
    agent_type="DQN",
    environment_config=env_config,
    training_config=training_config,
    agent_config=double_dqn_agent_config,
    save_dir="experiments/results/double_dqn",
)

# 开始训练
results_double_dqn = trainer_double_dqn.train_agent()

print("\n✓ Double DQN 训练完成")
print(f"训练轮数: {len(results_double_dqn['episode_rewards'])}")
print(f"最终奖励: {results_double_dqn['episode_rewards'][-1]:.2f}")

## 5. 性能对比可视化

绘制四个关键指标的对比图：
1. **Episode 奖励** - 训练进展
2. **订单完成率** - 任务成功率
3. **平均配送距离** - 效率指标
4. **训练损失** - 学习稳定性

In [ ]:
# 设置绘图样式
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Standard DQN vs Double DQN Performance Comparison", fontsize=16, fontweight="bold")

# 1. Episode 奖励曲线
ax = axes[0, 0]
if "episode_rewards" in results_dqn:
    ax.plot(results_dqn["episode_rewards"], label="Standard DQN", alpha=0.7, linewidth=2)
if "episode_rewards" in results_double_dqn:
    ax.plot(results_double_dqn["episode_rewards"], label="Double DQN", alpha=0.7, linewidth=2)
ax.set_xlabel("Episode")
ax.set_ylabel("Total Reward")
ax.set_title("Episode Rewards Comparison")
ax.legend()
ax.grid(True, alpha=0.3)

# 2. 订单完成率曲线
ax = axes[0, 1]
if "completion_rates" in results_dqn:
    ax.plot(results_dqn["completion_rates"], label="Standard DQN", alpha=0.7, linewidth=2)
if "completion_rates" in results_double_dqn:
    ax.plot(results_double_dqn["completion_rates"], label="Double DQN", alpha=0.7, linewidth=2)
ax.set_xlabel("Episode")
ax.set_ylabel("Completion Rate (%)")
ax.set_title("Order Completion Rate Comparison")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 105])

# 3. 平均配送距离对比
ax = axes[1, 0]
if "avg_distances" in results_dqn:
    ax.plot(results_dqn["avg_distances"], label="Standard DQN", alpha=0.7, linewidth=2)
if "avg_distances" in results_double_dqn:
    ax.plot(results_double_dqn["avg_distances"], label="Double DQN", alpha=0.7, linewidth=2)
ax.set_xlabel("Episode")
ax.set_ylabel("Average Distance")
ax.set_title("Average Delivery Distance Comparison")
ax.legend()
ax.grid(True, alpha=0.3)

# 4. 训练损失曲线（平滑）
ax = axes[1, 1]
window = 10
if "losses" in results_dqn and len(results_dqn["losses"]) > window:
    dqn_losses_smooth = np.convolve(
        results_dqn["losses"], np.ones(window) / window, mode="valid"
    )
    ax.plot(dqn_losses_smooth, label="Standard DQN", alpha=0.7, linewidth=2)
if "losses" in results_double_dqn and len(results_double_dqn["losses"]) > window:
    double_dqn_losses_smooth = np.convolve(
        results_double_dqn["losses"], np.ones(window) / window, mode="valid"
    )
    ax.plot(double_dqn_losses_smooth, label="Double DQN", alpha=0.7, linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("Training Loss Comparison (Smoothed)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()

# 保存图表
output_dir = Path("experiments/results")
output_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(output_dir / "dqn_comparison.png", dpi=300, bbox_inches="tight")
print(f"✓ 对比图表已保存: {output_dir / 'dqn_comparison.png'}")

plt.show()

## 6. 结果分析

In [ ]:
def calculate_stats(results, name):
    """计算统计指标"""
    if "episode_rewards" not in results or not results["episode_rewards"]:
        return {}
    
    last_100_rewards = results["episode_rewards"][-100:]
    last_100_completion = results.get("completion_rates", [])[-100:]
    last_100_distances = results.get("avg_distances", [])[-100:]
    
    return {
        "algorithm": name,
        "total_episodes": len(results["episode_rewards"]),
        "final_avg_reward": float(np.mean(last_100_rewards)) if last_100_rewards else 0,
        "std_reward": float(np.std(last_100_rewards)) if last_100_rewards else 0,
        "best_reward": float(max(results["episode_rewards"])),
        "final_avg_completion_rate": float(np.mean(last_100_completion)) if last_100_completion else 0,
        "final_avg_distance": float(np.mean(last_100_distances)) if last_100_distances else 0,
    }

# 计算统计数据
dqn_stats = calculate_stats(results_dqn, "Standard DQN")
ddqn_stats = calculate_stats(results_double_dqn, "Double DQN")

# 计算改进幅度
improvement = {}
if dqn_stats.get("final_avg_reward", 0) > 0:
    improvement["reward"] = (
        (ddqn_stats["final_avg_reward"] - dqn_stats["final_avg_reward"])
        / dqn_stats["final_avg_reward"]
        * 100
    )
if dqn_stats.get("final_avg_distance", 0) > 0:
    improvement["distance"] = (
        (dqn_stats["final_avg_distance"] - ddqn_stats["final_avg_distance"])
        / dqn_stats["final_avg_distance"]
        * 100
    )

# 打印结果
print("=" * 70)
print("实验结果摘要 (最近 100 episodes)")
print("=" * 70)

print("\n📊 Standard DQN:")
print(f"  训练轮数: {dqn_stats.get('total_episodes', 0)}")
print(f"  平均奖励: {dqn_stats.get('final_avg_reward', 0):.2f} ± {dqn_stats.get('std_reward', 0):.2f}")
print(f"  最佳奖励: {dqn_stats.get('best_reward', 0):.2f}")
print(f"  完成率: {dqn_stats.get('final_avg_completion_rate', 0):.1f}%")
print(f"  平均距离: {dqn_stats.get('final_avg_distance', 0):.2f}")

print("\n📊 Double DQN:")
print(f"  训练轮数: {ddqn_stats.get('total_episodes', 0)}")
print(f"  平均奖励: {ddqn_stats.get('final_avg_reward', 0):.2f} ± {ddqn_stats.get('std_reward', 0):.2f}")
print(f"  最佳奖励: {ddqn_stats.get('best_reward', 0):.2f}")
print(f"  完成率: {ddqn_stats.get('final_avg_completion_rate', 0):.1f}%")
print(f"  平均距离: {ddqn_stats.get('final_avg_distance', 0):.2f}")

if improvement:
    print("\n📈 改进幅度:")
    if "reward" in improvement:
        print(f"  奖励: {improvement['reward']:+.1f}%")
    if "distance" in improvement:
        print(f"  距离优化: {improvement['distance']:+.1f}%")

print("=" * 70)

## 7. 保存实验结果

In [ ]:
# 保存对比数据
comparison_data = {
    "experiment": "DQN vs Double DQN Comparison",
    "environment_config": env_config,
    "training_config": training_config,
    "results": {
        "standard_dqn": dqn_stats,
        "double_dqn": ddqn_stats,
    },
    "improvement": improvement,
}

output_file = output_dir / "comparison_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(comparison_data, f, indent=2, ensure_ascii=False)

print(f"✓ 对比结果已保存: {output_file}")
print(f"✓ 图表已保存: {output_dir / 'dqn_comparison.png'}")
print(f"\n📁 所有结果文件位于: {output_dir}")

## 8. 结论

### 主要发现

1. **收敛速度**: 
   - 观察奖励曲线的收敛趋势
   - Double DQN 理论上应更稳定

2. **性能指标**:
   - 比较最终的完成率和平均距离
   - 评估实际应用价值

3. **训练稳定性**:
   - 损失曲线的波动情况
   - Double DQN 对过估计的改善

### 后续改进方向

- 增加训练轮数以充分收敛
- 调整奖励函数提高完成率
- 尝试其他算法（PPO, A3C等）
- 优化网络结构和超参数

---

**实验完成时间**: 查看上方输出

**数据文件**: `experiments/results/`